# Building a new combinatorial algebra with 2x2 matrices

This notebook is meant as a guided example of how a user can create a new algebra from a mathematical idea.

In [ ]:
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "algcom").exists():
        repo_root = candidate
        break
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

import numpy as np
from algcom import Algebra, SparseVector
from fractions import Fraction

## 1. The "sparse" space of square matrices.

We want to work with 2x2 matrices with the usual multiplication. Let us remember that a matrix is a linear combination of the four elements $e_{11}$, $e_{12}$, $e_{21}$ and $e_{22}$. For instance, we can write the identity matrix as the sum of two vectors. 

In [ ]:
e11 = (1,1)
e12 = (1,2) 
e21 = (2,1)
e22 = (2,2)

identity = SparseVector(e11) + SparseVector(e22)
print(f"{identity}")

The complex numer $\mathtt{i}$, is given by $\mathtt{i} = e_{21} - e_{12}$.

In [ ]:
im = SparseVector(e21) - SparseVector(e12)
print(f"Imaginary unit: {im}")

Let us see how linear combinations are evaluated. Notice that we are using rationals for the coefficients.

In [ ]:
print(f"1 + 2i = {identity + 2*im}")
print(f"(1/3) + (1/2)i = {identity*Fraction(1,3) + im*Fraction(1,2)}") 

The printing of a linear expression is made such that the coefficients inside the delimiters ⟅...⟆ were always integers. The common denominator is expelled outside, such as the $...⟆/6$ in the last expression.

## Multiplication and defining an algebra 

In the space of matrices, and this means all the matrices, the produc is given by the rule
$$ e_{i,j} \cdot e_{k,l} = \delta_{j,k} e_{i,l}.$$
Here $\delta$ is Kronecker's delta and $i,j,k,l$ are any positive integers.

Let us implement the rule and execute some examples. 

Remember, the rule is given at the level of basis' elements!

In [ ]:
mult = lambda left,right : (left[0],right[1]) if left[1]==right[0] else 0 

print(f"Is {e12} = {mult(e11,e12)}?")
print(f"And {mult(e21,e22)} must be zero.")

We need now to lift this (set)-product into a bilinear multiplication. The class Algebra handles that bilinearity without much of a hassle for the user. 

A comment, though. Since the package was thought for combinatorial objects, the product should return a either list of elements of the basis or a sparse vector.
In this case, the product generates only one element of the basis.

In [ ]:
mult = lambda left,right : [ (left[0],right[1]) ] if left[1]==right[0] else []

sqmat = Algebra.from_rule( mult , one = identity ) 

And that's it. The moment of the truth has arrived.

In [ ]:
print(f"1 + i*i = {identity + sqmat.m(im,im)}")

And we can do more than that. We can compute for instance exponentials and logarithms without further implementation.
In the next example, we compute up to three digits.
$$ \exp\left\{ \log{2} + \frac{\pi}{4}\mathtt{i} \right\}  = \sqrt{2} + \sqrt{2}\mathtt{i} $$

In [ ]:
log2 = Fraction(np.log(2)).limit_denominator()
angle = Fraction(np.pi/4).limit_denominator()

print(f"log(2*1)={sqmat.logarithm(2*identity,order=750)}" )

operando = sqmat.unit(log2) + angle * im 
result =  sqmat.exponential(  operando , order=20)
print(f"exp( log(2) + pi/4 * i )={result}")

print(f"Geomentric series convergence: {sqmat.geometric(sqmat.unit(Fraction(1,2)),order=9)}")

The important idea is that one does not need to write a new class for every algebra. Once defined the product rule, specify the unit (via the "one"), and the rest of the algebraic operations become available.

## Other linear constructions



One might want to work in the dual space. In the case of the matrices as we are working with them now, the functionals (in the dual basis) are the ones that detect each of the entries.

In [ ]:
angle = Fraction(np.pi/6).limit_denominator() 
print(f"Sine of the angle: {round(np.sin(float(angle)),3)}")

rotation = sqmat.exponential( angle * im )
basis_elem = SparseVector({e21 : 1})
element_of_the_dual_basis = basis_elem.dual()
evaluation = element_of_the_dual_basis(rotation)
print(f"Entry {e21} of the rotation: {evaluation}")

From here, it is obvious that the trace can be computed using the dual of the identity matrix.

In [ ]:
trace = identity.dual()

print(f"Trace of i: {trace(im)}")

easy_example = SparseVector({ e11: 1, e12 : 2 , e21 : 3 , e22 :4 })
print(f"Trace of {easy_example}: {trace(easy_example)}")

angle = Fraction(np.pi/4).limit_denominator() 
rotation = sqmat.exponential( angle* im )
evaluation = trace( rotation )
print(f"Trace of the rotation pi/4: { round(float(evaluation),3) }")

However, and here lies the real use of this module, the trace and the product can be defined infinitely. And the identity matrix cannot, because is not a finite object. 

In [ ]:
A = SparseVector({ (256,343) : 5} )
B = SparseVector({ (343,256) : Fraction(1,5) , (343,121) : 3 }) 

print(f"AB={sqmat.m(A,B)}")
print(f"AB-BA={sqmat.commutator(A,B)}")

Trace = lambda _A : sum( c for k,c in _A._data.items() if k.value[0]==k.value[1] )
print(f"Trace(AB)={Trace(sqmat.m(A,B))}")
print(f"Trace(AB-BA)={Trace(sqmat.commutator(A,B) )}")